# Concrete Strength — Optuna & Hyperopt

**Task:** Build a Deep Learning regression model for the Concrete Compressive Strength dataset, then tune the model with **Optuna** and **Hyperopt** and train a final model using the best parameters.

The notebook is self-contained: it downloads the dataset, preprocesses it, builds the neural network, performs tuning, evaluates the final model, and saves the best parameters/results.

In [ ]:
# Install if needed:
# !pip install -q tensorflow optuna hyperopt pandas scikit-learn matplotlib seaborn openpyxl

import os, random, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import optuna
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

## 1. Load the dataset

The dataset is the UCI **Concrete Compressive Strength** dataset. It contains 8 input features and the concrete compressive strength as the regression target.

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/concrete/compressive/Concrete_Data.xls"
df = pd.read_excel(url)

df.columns = [
    "cement", "blast_furnace_slag", "fly_ash", "water",
    "superplasticizer", "coarse_aggregate", "fine_aggregate",
    "age", "strength"
]

print(df.shape)
display(df.head())
display(df.describe().T)

In [ ]:
X = df.drop(columns="strength").astype("float32")
y = df["strength"].astype("float32")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED
)

# A validation split is kept separate from the test set.
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.20, random_state=SEED
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

print("Train:", X_train_s.shape, "Validation:", X_val_s.shape, "Test:", X_test_s.shape)

## 2. Baseline Deep Learning model

In [ ]:
def build_baseline_model():
    model = keras.Sequential([
        layers.Input(shape=(X_train_s.shape[1],)),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.20),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.10),
        layers.Dense(1)
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse",
        metrics=[keras.metrics.MeanAbsoluteError(name="mae")]
    )
    return model

baseline = build_baseline_model()
baseline_history = baseline.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=100,
    batch_size=32,
    verbose=0,
    callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=12, restore_best_weights=True)]
)

baseline_pred = baseline.predict(X_test_s, verbose=0).ravel()
baseline_metrics = {
    "MAE": mean_absolute_error(y_test, baseline_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, baseline_pred)),
    "R2": r2_score(y_test, baseline_pred)
}
baseline_metrics

## 3. Optuna hyperparameter tuning

We tune:
- number of hidden layers
- units per layer
- dropout
- learning rate
- batch size

The objective minimizes validation MSE.

In [ ]:
def build_concrete_model(params):
    model = keras.Sequential()
    model.add(layers.Input(shape=(X_train_s.shape[1],)))

    for i in range(params["n_layers"]):
        model.add(layers.Dense(
            params[f"units_{i}"],
            activation=params["activation"]
        ))
        if params[f"dropout_{i}"] > 0:
            model.add(layers.Dropout(params[f"dropout_{i}"]))

    model.add(layers.Dense(1))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=params["learning_rate"]),
        loss="mse",
        metrics=[keras.metrics.MeanAbsoluteError(name="mae")]
    )
    return model

def optuna_objective(trial):
    tf.keras.backend.clear_session()

    n_layers = trial.suggest_int("n_layers", 1, 3)
    params = {
        "n_layers": n_layers,
        "activation": trial.suggest_categorical("activation", ["relu", "tanh"]),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 5e-3, log=True),
    }

    for i in range(n_layers):
        params[f"units_{i}"] = trial.suggest_int(f"units_{i}", 16, 256, step=16)
        params[f"dropout_{i}"] = trial.suggest_float(f"dropout_{i}", 0.0, 0.40, step=0.05)

    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128])

    model = build_concrete_model(params)
    history = model.fit(
        X_train_s, y_train,
        validation_data=(X_val_s, y_val),
        epochs=80,
        batch_size=batch_size,
        verbose=0,
        callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)]
    )

    return float(min(history.history["val_loss"]))

optuna_study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
optuna_study.optimize(optuna_objective, n_trials=30, show_progress_bar=False)

print("Optuna best validation MSE:", optuna_study.best_value)
print("Optuna best parameters:")
print(optuna_study.best_params)

In [ ]:
optuna_results = pd.DataFrame(optuna_study.trials_dataframe())
display(optuna_results.sort_values("value").head(10))

## 4. Hyperopt tuning

In [ ]:
def hyperopt_objective(params):
    tf.keras.backend.clear_session()

    n_layers = int(params["n_layers"])
    model_params = {
        "n_layers": n_layers,
        "activation": params["activation"],
        "learning_rate": float(params["learning_rate"]),
    }

    for i in range(n_layers):
        model_params[f"units_{i}"] = int(params[f"units_{i}"])
        model_params[f"dropout_{i}"] = float(params[f"dropout_{i}"])

    batch_size = int(params["batch_size"])
    model = build_concrete_model(model_params)

    history = model.fit(
        X_train_s, y_train,
        validation_data=(X_val_s, y_val),
        epochs=80,
        batch_size=batch_size,
        verbose=0,
        callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)]
    )

    best_val = float(min(history.history["val_loss"]))
    return {"loss": best_val, "status": STATUS_OK}

space = {
    "n_layers": hp.choice("n_layers", [1, 2, 3]),
    "activation": hp.choice("activation", ["relu", "tanh"]),
    "learning_rate": hp.loguniform("learning_rate", np.log(1e-4), np.log(5e-3)),
    "batch_size": hp.choice("batch_size", [16, 32, 64, 128]),
    # The conditional parameters are interpreted according to n_layers.
    "units_0": hp.quniform("units_0", 16, 256, 16),
    "units_1": hp.quniform("units_1", 16, 256, 16),
    "units_2": hp.quniform("units_2", 16, 256, 16),
    "dropout_0": hp.quniform("dropout_0", 0.0, 0.40, 0.05),
    "dropout_1": hp.quniform("dropout_1", 0.0, 0.40, 0.05),
    "dropout_2": hp.quniform("dropout_2", 0.0, 0.40, 0.05),
}

hyperopt_trials = Trials()
best_hyperopt = fmin(
    fn=hyperopt_objective,
    space=space,
    algo=tpe.suggest,
    max_evals=30,
    trials=hyperopt_trials,
    rstate=np.random.default_rng(SEED)
)

print("Hyperopt raw best result:")
print(best_hyperopt)
print("Best Hyperopt validation MSE:", min(t["result"]["loss"] for t in hyperopt_trials.trials))

## 5. Train final model with Optuna best parameters

For the final model we use the best Optuna configuration and evaluate only once on the untouched test set.

In [ ]:
best = optuna_study.best_params.copy()
batch_size = best.pop("batch_size")

final_model = build_concrete_model(best)
final_history = final_model.fit(
    X_train_s, y_train,
    validation_data=(X_val_s, y_val),
    epochs=150,
    batch_size=batch_size,
    verbose=0,
    callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)]
)

final_pred = final_model.predict(X_test_s, verbose=0).ravel()
final_metrics = {
    "MAE": mean_absolute_error(y_test, final_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, final_pred)),
    "R2": r2_score(y_test, final_pred)
}

comparison = pd.DataFrame([baseline_metrics, final_metrics], index=["Baseline", "Optuna tuned"])
display(comparison)

plt.figure(figsize=(7,5))
plt.scatter(y_test, final_pred, alpha=0.6)
lims = [min(y_test.min(), final_pred.min()), max(y_test.max(), final_pred.max())]
plt.plot(lims, lims, "--")
plt.xlabel("Actual strength")
plt.ylabel("Predicted strength")
plt.title("Concrete: Actual vs Predicted")
plt.show()

In [ ]:
with open("concrete_best_params_optuna.json", "w") as f:
    json.dump(optuna_study.best_params, f, indent=2)

final_model.save("concrete_final_optuna.keras")

print("Saved: concrete_best_params_optuna.json")
print("Saved: concrete_final_optuna.keras")

### Final takeaway

The notebook demonstrates the complete workflow from dataset loading → preprocessing → baseline DL model → Optuna tuning → Hyperopt tuning → final model → test evaluation.

**Important:** the exact best parameters and scores are generated when the notebook is executed, because tuning is stochastic and depends on the runtime/environment.